# BagHunter reproducible fixture research

This notebook executes the frozen local fixture pipeline and inspects its immutable reports.
Every security, price, filing and outcome below is synthetic. It is software acceptance,
not empirical investment evidence, a verified thesis, a forecast or an untouched holdout.
No credentials, provider requests, subscriptions, broker actions or deployment are involved.

Open with a Python environment installed from the repository's uv.lock, or execute every cell
without a notebook server using `uv run python scripts/run_research_notebook.py` from the repo.
Outputs intentionally remain uncommitted. The runner saves a transcript and hashed execution receipt.


In [ ]:
import hashlib
import json
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

research_root = Path.cwd()
if not (research_root / "pyproject.toml").is_file():
    research_root = research_root.parent
assert (research_root / "packages/pipeline.py").is_file(), "Start inside this repository"
sys.path.insert(0, str(research_root))

In [ ]:
import pyarrow.parquet as pq

from packages.data.run_archive import verify_integrity
from packages.pipeline import DEFAULT_ASOF, run


def show(value):
    print(json.dumps(value, indent=2, sort_keys=True, allow_nan=False))


output = research_root / "artifacts/notebook/fixture/latest"
configuration = {"asof": DEFAULT_ASOF, "seed": 7, "mode": "FIXTURE"}
show(configuration)

## Frozen source, configuration and repeatability

The run key binds source/code/policy hashes, as-of time and seed. Repeating the same configuration
must reuse the verified archive. The original raw source remains separate from transformed tables.


In [ ]:
summary = run(output, datetime.fromisoformat(configuration["asof"]), seed=configuration["seed"])
folder = output / "runs" / summary["run_id"]
verify_integrity(folder)
research = json.loads((folder / "run.json").read_text(encoding="utf-8"))
assert research["mode"] == "FIXTURE"
raw_path = output.parent / "raw" / (research["sources"][0]["sha256"] + ".json")
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == research["sources"][0]["sha256"]
again = run(output, datetime.fromisoformat(configuration["asof"]), seed=configuration["seed"])
assert again["cache_reused"] and again["run_id"] == research["run_id"]
show(
    {
        "run_id": research["run_id"],
        "code_sha256": research["code_sha256"],
        "source_sha256": research["sources"][0]["sha256"],
        "discovery_policy": research["discovery"]["policy"],
        "portfolio_policy": research["portfolio"]["policy"],
    }
)

## Knowable features and filing cutoff

Check the later acceptance/publication time plus the declared processing lag against each
feature's own research clock. Hindsight outcomes are not passed to feature construction.
Unknown factors remain null; an unvalidated score is not a calibrated probability.


In [ ]:
for signal in research["signals"]:
    clock = datetime.fromisoformat(signal["asof_time"]).astimezone(UTC)
    for fact in signal["evidence"]:
        known = max(
            datetime.fromisoformat(fact["accepted_at"]).astimezone(UTC),
            datetime.fromisoformat(fact["published_at"]).astimezone(UTC),
        )
        assert known + timedelta(hours=signal.get("processing_lag_hours", 24)) <= clock
columns = set(pq.read_schema(folder / "gold/features.parquet").names)
assert not {"max_multiple", "terminal_total_return", "outcome", "outcome_group"} & columns
show(
    [
        {
            key: row.get(key)
            for key in ("security_id", "asof_time", "stage", "score", "completeness", "components")
        }
        for row in research["signals"]
    ]
)

## Hindsight cohort and matched failure — separate research namespace

These later outcomes are descriptive labels, not information that was available to predictors.
Maximum attained wealth is different from terminal holding return. Missing horizons stay missing;
known delist proceeds are explicit. Retrospective matching weights are not population probabilities.


In [ ]:
hindsight_labels = research["study"]["labels"]
show(
    {
        "anchor": research["study"]["anchor"],
        "scope": research["study"]["scope"],
        "eligible_denominator": research["study"]["denominator"],
        "pairs": research["study"]["pairs"],
        "later_outcomes": hindsight_labels,
    }
)

## Frozen screening and constrained portfolio diagnostics

The fixture has already been inspected. Chronological partitions and purge/embargo mechanics do
not make it an untouched real holdout. Costs, whole shares, cash, capacity and concentration are
model assumptions. No return displayed here establishes real alpha or executable capacity.


In [ ]:
show(
    {
        "screening_period": [
            research["backtest"]["period_start"],
            research["backtest"]["period_end"],
        ],
        "execution": research["backtest"]["execution"],
        "partitions": research["validation"]["partitions"],
        "holdout_status": research["validation"]["holdout_status"],
        "screening_metrics": research["validation"]["metrics"],
        "daily_portfolio_risk": research["portfolio"]["daily_risk"],
        "executions": len(research["portfolio"]["executions"]),
        "exclusions": len(research["portfolio"]["exclusions"]),
    }
)

## Every declared stress retained

Highest-contribution removals are ex-post diagnostics. Unavailable adjusted prices and rejected
policies receive no substituted metric. There is no selected winning variant or empirical study.


In [ ]:
stress = research["validation"]["robustness"]
assert len(stress["attempts"]) == 12
assert stress["dataset_sha256"] == research["sources"][0]["sha256"]
assert stress["attempts"][-1]["report"] is None
show(
    [
        {
            "attempt": row["id"],
            "scope": row["selection_scope"],
            "status": row["status"],
            "reason": row["reason"],
            "fixture_total_return": row["report"]["metrics"]["total_return"]
            if row["report"]
            else None,
        }
        for row in stress["attempts"]
    ]
)
show(
    {
        "verified_real_theses": 0,
        "limitations": research["limitations"],
        "stress_limitations": stress["limitations"],
        "reports": sorted(path.name for path in (folder / "reports").iterdir() if path.is_file()),
    }
)

## Real-data boundary

Finviz observations and SEC entity evidence remain separately quarantined. This notebook does not
promote security identities, certify original filing vintages, fill action/delist gaps or infer
historical universe membership. See docs/SOURCES.md, docs/MODEL_CARD.md and the current STATUS.md.
Representative real histories, benchmarks and independent factual critique are still needed for
empirical research conclusions. A useful fixture notebook does not complete those requirements.
